# 🧠 SONATA Edge Editor — Demo

This notebook demonstrates the `SonataEdgeEditor` context manager.

**Workflow:**
1. Select a SONATA edge HDF5 file
2. Select an edge type to edit
3. Choose what fraction of edges to randomly replace
4. Run the replacement and compare before/after histograms

In [1]:
import polars as pl
import numpy as np
import altair as alt
import tempfile
import os
import h5py
from pathlib import Path
from airavata_cerebrum.sonata.edge import SonataEdgeEditor, EditConfig, IndexType

## Step 1 — Select Input Edges File

In [2]:
INPUT_H5 = "model_prior/v1_test/network_v1/left_local_edges.edge_types_120.sorted.h5"

## Step 2 — Inspect Edges File Properties

In [3]:
et_cfg = EditConfig(
        edge_type_index_group = "edge_type_to_index",
        type_to_range_dataset = "node_id_to_range",
        range_to_edge_dataset = "range_to_edge_id",
)

with SonataEdgeEditor(INPUT_H5, edge_type_index_cfg=et_cfg) as _ed:
    _idx   = _ed.has_indices()
    _pop   = _ed.population

    # read full edge table to get type counts
    import h5py as _h5py
    with _h5py.File(INPUT_H5, "r") as _f:
        _grp = _f[f"edges/{_pop}"]
        import numpy as _np
        _etypes = _grp["edge_type_id"][:]

import polars as _pl
counts_df = (
    _pl.DataFrame({"edge_type_id": _etypes})
    .group_by("edge_type_id")
    .agg(_pl.len().alias("count"))
    .sort("edge_type_id")
)
_type_ids = counts_df["edge_type_id"].to_list()
# expose for downstream cells
file_population = _pop
file_type_ids   = _type_ids

pl.DataFrame(
    {
        "Edge Table Properties": ["Population", "Total Edges", "Unique Edge Types", "Node Index", "Edge Type Index" ],
         "Values" : [
             _pop, str(len(_etypes)), str(len(_type_ids)),
             "✅ present" if IndexType.NODE in _idx else "❌ absent",
             "✅ present" if IndexType.EDGE_TYPE in _idx else "❌ absent"
         ]
    }
)

Edge Table Properties,Values
str,str
"""Population""","""left_local"""
"""Total Edges""","""164024874"""
"""Unique Edge Types""","""120"""
"""Node Index""","""❌ absent"""
"""Edge Type Index""","""✅ present"""


## Step 3 — Edge-type distribution *before* editing

In [4]:
chart_before = (
    alt.Chart(counts_df.to_pandas().head(15), title="Edge counts by type — BEFORE")
    .mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4)
    .encode(
        x=alt.X("edge_type_id:O", title="Edge type ID", axis=alt.Axis(labelAngle=0)),
        y=alt.Y(
           "count:Q",
            title="Number of edges", 
            scale=alt.Scale(type="linear"),
            axis=alt.Axis(format="~s")
        ),
        color=alt.Color(
                "edge_type_id:O",
                scale=alt.Scale(scheme="tableau10"),
                legend=None,
        ),
        tooltip=["edge_type_id:O", "count:Q"],
    )
    .properties(width=420, height=280)
)
chart_before

alt.Chart(...)

## Step 4 -- Parameters for Selecting a fraction of Edge
1. Provide the edge type id
2. Provide the fraction of edges to retain (randomly sampled from the input)

In [5]:
EDGE_TYPE_TO_REMOVE   =  111
EDGE_FRACTION_TO_RETAIN   = float(0.4)

## Step 5 -- Run the code for editing edges


In [6]:
_etype = EDGE_TYPE_TO_REMOVE
_frac = EDGE_FRACTION_TO_RETAIN
# write output to a temp file so we never modify the source
_out_tmp    = tempfile.NamedTemporaryFile(suffix=".h5", delete=False)
_out_tmp.close()
out_h5_path = Path(_out_tmp.name)

with SonataEdgeEditor(INPUT_H5, edge_type_index_cfg=et_cfg) as _ed:
    _edges = _ed.get_edges_by_type(_etype)
    _n_total = len(_edges)

    # sample a random subset using polars — fraction argument avoids manual n calculation
    _replacement = _edges.sample(fraction=_frac, shuffle=True)

    _ed.replace_edges_by_type(
        edge_type_id=_etype,
        new_edges=_replacement,
        output_path=out_h5_path,
    )

_summary = (
    f"**Edge type `{_etype}`:** original {_n_total} edges → "
    f"replaced with a random subset of **{len(_replacement)}** edges "
    f"({int(_frac * 100)}% of the original set)."
)
print(_summary)

**Edge type `111`:** original 71916 edges → replaced with a random subset of **28766** edges (40% of the original set).


## Step 6 -- Comparing before and after edge population distribution

The reduction in the population of ege type 111 is shown below

In [7]:
# read type counts from output file
with h5py.File(out_h5_path, "r") as _f:
    _pop_after = list(_f["edges"].keys())[0]
    _etypes_after = _f[f"edges/{_pop_after}/edge_type_id"][:]

counts_after = (
    pl.DataFrame({"edge_type_id": _etypes_after})
    .group_by("edge_type_id")
    .agg(pl.len().alias("count"))
    .sort("edge_type_id")
)

chart_after = (
    alt.Chart(counts_after.to_pandas().head(15), title="Edge counts by type — AFTER")
    .mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4)
    .encode(
        x=alt.X("edge_type_id:O", title="Edge type ID", axis=alt.Axis(labelAngle=0)),
        y=alt.Y(
            "count:Q", title="Number of edges",
            #scale=alt.Scale(type="log", base=10, domainMin=1),
            axis=alt.Axis(format="~s")
        ),
        color=alt.Color(
            "edge_type_id:O",
            scale=alt.Scale(scheme="tableau10"),
            legend=None,
        ),
        tooltip=["edge_type_id:O", "count:Q"],
    )
    .properties(width=420, height=280)
)
alt.hconcat(chart_before, chart_after)

alt.HConcatChart(...)